# HyDE — Hypothetical Document Embeddings
## Imagine the Answer First — Hypothetical Document Embeddings
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/55-hyde-rag/hyde_workbook.ipynb)

Standard RAG embeds the *user query* and searches for similar documents. But queries are short, terse, and live in a different embedding space than the longer, information-dense documents they're meant to retrieve. **HyDE** (Hypothetical Document Embeddings) fixes this by asking the LLM to first generate a *hypothetical answer*, then embedding that answer for retrieval — because a hypothetical answer lives in the same embedding space as real documents. By the end of this workshop you will understand *why* the query-document embedding gap exists, *how* HyDE bridges it, and *when* it helps vs hurts.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — The query-document embedding gap |
| 2 | **Hypothesis generation** — LLM writes a draft answer |
| 3 | **Embedding comparison** — query vec vs hypothesis vec |
| 4 | **Retrieval comparison** — standard RAG vs HyDE |
| 5 | **Full Pipeline** — hypothesize → retrieve → generate with LangGraph |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with `.venv` and `requirements.txt` installed, **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Gao, L., Ma, X., Lin, J., & Callan, J. (2022). *Precise Zero-Shot Dense Retrieval without Relevance Labels.* ACL 2023. https://arxiv.org/abs/2212.10496

In [ ]:
# Install dependencies — runs only on Google Colab.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langchain-chroma",
        "chromadb",
        "langgraph",
        "numpy",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM or embedding cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — The Query-Document Embedding Gap

---

### Why Standard Retrieval Struggles with Short Queries

Consider this query:

> *"How does RAG reduce hallucination?"*

And this document:

> *"RAG grounds LLM responses by retrieving relevant documents before generation, ensuring the model cites factual sources instead of relying solely on its training data, which reduces fabricated content."*

Both cover the same topic, but:
- The query is 7 words — dense, terse, question-form
- The document is 30 words — explanatory, statement-form, vocabulary-rich

Embedding models learn to place *similar text* close together. A short question and a long answer, while semantically related, may land further apart in the embedding space than two documents covering the same topic in the same style.

---

### The HyDE Solution

```
Standard RAG:                          HyDE:

query ──► embed ──► search             query ──► LLM ──► hypothesis
           │                                              │
   "How does RAG...?"              "RAG reduces hallucination by..."  ← document-like
           │                                              │
           ▼                                              ▼ embed
    [0.12, -0.45, ...]                           [0.31, 0.28, ...]   ← closer to docs
           │                                              │
           ▼                                              ▼
     cosine search                              cosine search
```

The hypothesis is *intentionally imprecise* — it may be factually wrong. That is fine. Its job is not to answer the question but to occupy the *same embedding neighbourhood* as correct documents.

---

### When HyDE Helps vs Hurts

| Scenario | HyDE helps? | Why |
|----------|------------|-----|
| Open-domain factual questions | Yes | Hypothesis closes the query-document style gap |
| Technical / niche domains | Yes | LLM approximates domain vocabulary even if facts are wrong |
| Very short queries (1–3 words) | Yes | Hypothesis adds semantic richness |
| Keyword / exact-match queries | No | Hypothesis may add noise that hurts recall |
| Time-sensitive facts (latest news) | No | LLM may hallucinate outdated details that mislead retrieval |
| Queries already document-like | No | Little benefit; adds LLM latency |

**Rule of thumb:** HyDE improves retrieval when the query vocabulary is sparse relative to the document vocabulary. Always benchmark both approaches on your corpus.

In [ ]:
# Corpus used throughout the workshop

SAMPLE_DOCS = [
    "RAG grounds LLM responses by retrieving relevant documents before generation.",
    "The retriever fetches documents that match the query using vector similarity.",
    "Hallucination occurs when a language model invents facts not in its training data.",
    "Embeddings convert text to dense vectors so similar meanings cluster together.",
    "ChromaDB is a lightweight vector store that runs locally without a server.",
    "OpenAI text-embedding-3-small produces 1536-dimensional embeddings efficiently.",
    "Hypothetical Document Embeddings (HyDE) embeds a generated answer, not the query.",
    "A bi-encoder embeds query and document separately for fast approximate search.",
    "Cosine similarity measures how similar two embedding vectors are regardless of length.",
    "Prompt engineering shapes model outputs by crafting precise instruction text.",
]

QUERIES = [
    "How does retrieval-augmented generation reduce hallucination?",
    "What is the difference between a bi-encoder and a cross-encoder?",
    "Why does embedding a hypothetical answer sometimes outperform embedding the query?",
]

print(f"Corpus: {len(SAMPLE_DOCS)} documents")
for i, d in enumerate(SAMPLE_DOCS):
    print(f"  [{i:02d}] {d}")

---

## Part 2 — Hypothesis Generation

---

The first node in HyDE asks the LLM to write a single-paragraph draft answer. The prompt is carefully designed:

1. **Request a paragraph** — not a sentence. Longer text produces richer embeddings.
2. **Explicitly allow hypothetical details** — the model shouldn't struggle to fill gaps. Imprecision is acceptable.
3. **State the goal** — the hypothesis will be used for retrieval, not shown to the user.

```python
HYPOTHESIS_PROMPT = (
    "Write a single-paragraph answer to the following question. "
    "It may contain hypothetical details — accuracy is not required. "
    "The goal is to produce text whose embedding is similar to a real answer."
)
```

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

HYPOTHESIS_PROMPT = (
    "Write a single-paragraph answer to the following question. "
    "It may contain hypothetical details — accuracy is not required. "
    "The goal is to produce text whose embedding is similar to a real answer."
)

# temperature=0 for deterministic hypotheses — reproducible embeddings across runs
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

hypotheses: dict[str, str] = {}

# Each hypothesis is cached in a dict keyed by the original query for later reuse
for query in QUERIES:
    msgs = [SystemMessage(content=HYPOTHESIS_PROMPT), HumanMessage(content=query)]
    hypotheses[query] = llm.invoke(msgs).content

# Show results
for query, hyp in hypotheses.items():
    print(f"QUERY: {query}")
    print(f"HYP  : {hyp[:200]}...")
    print(f"       ({len(hyp)} chars, {len(hyp.split())} words)")
    print()

---

## Part 3 — Embedding Comparison: Query vs Hypothesis

---

We embed all corpus documents, all queries, and all hypotheses, then measure:

1. **Query→Document similarity** — how close is the raw query to the top-1 relevant document?
2. **Hypothesis→Document similarity** — how close is the hypothesis to the top-1 relevant document?

If HyDE is working, the hypothesis should be *more similar* to the relevant document than the raw query.

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings


def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


# text-embedding-3-small: 1536-d, faster and cheaper than ada-002 — sufficient for this demo
emb_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed everything
# embed_documents batches all texts in one API call — prefer over looping embed_query
doc_vecs  = emb_model.embed_documents(SAMPLE_DOCS)
query_vecs = emb_model.embed_documents(QUERIES)  # same model for fair comparison
hyp_vecs  = emb_model.embed_documents(list(hypotheses.values()))

print(f"Embedded: {len(doc_vecs)} docs, {len(query_vecs)} queries, {len(hyp_vecs)} hypotheses")
print(f"Vector dimension: {len(doc_vecs[0])}")

In [ ]:
# ─── For each query: top-1 doc by query-embed vs top-1 doc by hypothesis-embed ─

print(f"{'Query (truncated)':<45} {'Q→Doc':>7} {'H→Doc':>7} {'HyDE?':>7}")
print("-" * 72)

for i, (query, qvec, hvec) in enumerate(zip(QUERIES, query_vecs, hyp_vecs)):
    # Similarity of raw query to each doc
    q_sims = [cosine_sim(qvec, dv) for dv in doc_vecs]
    # Similarity of hypothesis to each doc
    h_sims = [cosine_sim(hvec, dv) for dv in doc_vecs]

    best_q_sim = max(q_sims)
    best_h_sim = max(h_sims)
    best_q_doc = SAMPLE_DOCS[q_sims.index(best_q_sim)][:40]
    best_h_doc = SAMPLE_DOCS[h_sims.index(best_h_sim)][:40]

    hyde_wins = "YES" if best_h_sim > best_q_sim else "NO"
    print(f"{query[:45]:<45} {best_q_sim:>7.3f} {best_h_sim:>7.3f} {hyde_wins:>7}")
    if best_q_doc != best_h_doc:
        print(f"  Q top doc: {best_q_doc}")
        print(f"  H top doc: {best_h_doc}")
    print()

---

## Part 4 — Retrieval Comparison: Standard RAG vs HyDE

---

Now we build the Chroma vector store and run both retrieval strategies:

- **Standard RAG:** embed query → search by vector similarity
- **HyDE:** embed hypothesis → search by vector similarity

We compare the retrieved documents side by side for each query.

In [ ]:
from langchain_chroma import Chroma

# from_texts embeds and stores in one call; no persist_directory = in-memory only
vectorstore = Chroma.from_texts(
    SAMPLE_DOCS,
    emb_model,
# collection_name isolates this index — prevents stale docs if the cell reruns
    collection_name="hyde-demo",
)
print(f"Vector store ready: {len(SAMPLE_DOCS)} documents indexed.")

In [ ]:
# ─── Side-by-side: standard RAG vs HyDE top-3 ────────────────────────────────

# k=3 is a common default — lower k raises precision, higher k raises recall
K = 3

for query, hypothesis in hypotheses.items():
    # Standard: embed query
    std_docs = vectorstore.similarity_search(query, k=K)

    # HyDE: embed hypothesis
    hyp_vec = emb_model.embed_query(hypothesis)
    hyde_docs = vectorstore.similarity_search_by_vector(hyp_vec, k=K)
# HyDE uses embed_query(hypothesis) then searches by vector — bypasses internal query embedding

    print(f"QUERY: {query}")
    print(f"HYPOTHESIS (first 100 chars): {hypothesis[:100]}...")
    print()
    print(f"  {'Standard RAG top-3':<45}  {'HyDE top-3'}")
    print(f"  {'─'*45}  {'─'*45}")
    for s, h in zip(std_docs, hyde_docs):
        same = " =" if s.page_content == h.page_content else ""
        print(f"  {s.page_content[:43]:<45}  {h.page_content[:43]}{same}")
    print()

---

## Part 5 — Full Pipeline with LangGraph

---

### State and Graph

```python
class HyDEState(TypedDict):
    query:      str         # user question
    hypothesis: str         # LLM-generated draft answer
    documents:  list[str]   # top-k retrieved by hypothesis embedding
    answer:     str         # final grounded answer
```

```
START
  │
  ▼
hypothesize   ─── LLM writes draft answer
  │
  ▼
retrieve      ─── embed hypothesis, search Chroma
  │
  ▼
generate      ─── LLM answers from retrieved context
  │
  ▼
 END
```

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


# TypedDict gives LangGraph typed access to state keys without runtime overhead
class HyDEState(TypedDict):
    query:      str
    hypothesis: str
    documents:  list
    answer:     str


def hypothesize_node(state: HyDEState) -> dict:
    msgs = [SystemMessage(content=HYPOTHESIS_PROMPT), HumanMessage(content=state["query"])]
    return {"hypothesis": llm.invoke(msgs).content}


def retrieve_node(state: HyDEState) -> dict:
    hyp_vec = emb_model.embed_query(state["hypothesis"])
    docs = vectorstore.similarity_search_by_vector(hyp_vec, k=3)
    return {"documents": [d.page_content for d in docs]}


def generate_node(state: HyDEState) -> dict:
    context = "\n".join(state["documents"])
    msg = HumanMessage(
        content=f"Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {state['query']}"
    )
    return {"answer": llm.invoke([msg]).content}


graph = StateGraph(HyDEState)
# StateGraph nodes return partial dicts — only the keys they update, not the full state
graph.add_node("hypothesize", hypothesize_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "hypothesize")
graph.add_edge("hypothesize", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
app = graph.compile()
# compile() locks the graph topology — edges can't be added after this point

print("Pipeline compiled: hypothesize → retrieve → generate")

In [ ]:
# ─── Run all three queries through the full HyDE pipeline ────────────────────

for query in QUERIES:
    print(f"{'=' * 65}")
    print(f"Query: {query}")
# Pass empty strings/lists for unused state keys — invoke requires all TypedDict fields
    result: HyDEState = app.invoke(
        {"query": query, "hypothesis": "", "documents": [], "answer": ""}
    )
    print(f"Hypothesis : {result['hypothesis'][:120]}...")
    print(f"Retrieved  : {len(result['documents'])} docs")
    print(f"Answer     : {result['answer']}")
    print()

---

### Exercise 1 — Measure the Embedding Distance

**Goal:** For the first query (`"How does retrieval-augmented generation reduce hallucination?"`), compute:

1. Cosine similarity between the **query vector** and each of the 10 corpus documents
2. Cosine similarity between the **hypothesis vector** and each of the 10 corpus documents
3. Print a ranked table showing both scores for every document

**Question:** Which document ranks #1 under the query embedding vs the hypothesis embedding? Are they the same?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

Q = QUERIES[0]
H = hypotheses[Q]

# TODO: embed Q and H with emb_model.embed_query()
# TODO: compute cosine_sim() of each against all doc_vecs
# TODO: print a table: rank, query_sim, hyp_sim, document (first 50 chars)
pass

### Exercise 2 — Try a Different Hypothesis Prompt

**Goal:** Replace `HYPOTHESIS_PROMPT` with a shorter, more directive version:

```
"Answer the following question in 2-3 sentences using technical vocabulary."
```

Run the third query (`"Why does embedding a hypothetical answer sometimes outperform embedding the query?"`) with both prompts. Compare:
- The length and style of each hypothesis
- The top-3 documents retrieved by each

**Question:** Does the shorter prompt produce a more or less useful hypothesis for retrieval?

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

ALT_PROMPT = "Answer the following question in 2-3 sentences using technical vocabulary."
Q = QUERIES[2]

# TODO: generate hypothesis with HYPOTHESIS_PROMPT and ALT_PROMPT for Q
# TODO: embed each hypothesis and retrieve top-3 docs
# TODO: print both hypotheses and their respective top-3 retrieved docs
pass

---

## What's Next?

You now understand why HyDE bridges the query-document embedding gap and how to implement it as a LangGraph pipeline.

### Related retrieval techniques
- **Example 56 — Contextual Compression** ([`../56-contextual-compression/`](../56-contextual-compression/README.md)): after retrieving docs, filter the retrieved chunks to only the passage that answers the query — pairs well with HyDE
- **Example 53 — Chunking Strategies** ([`../53-chunking-strategies/chunking_workbook.ipynb`](../53-chunking-strategies/chunking_workbook.ipynb)): chunking determines what's in your index — HyDE improves retrieval, but can't fix bad chunks
- **Example 54 — Reranking RAG** ([`../54-reranking-rag/reranking_workbook.ipynb`](../54-reranking-rag/reranking_workbook.ipynb)): combine HyDE (for recall) with cross-encoder reranking (for precision) for maximum retrieval quality

### Evaluate your improvement
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/`](../16-rag-eval-notebook/README.md)): measure whether HyDE actually improves Faithfulness and Context Recall on your corpus

### Key paper
- Gao et al. (2022). *Precise Zero-Shot Dense Retrieval without Relevance Labels.* ACL 2023. https://arxiv.org/abs/2212.10496

---

*Workshop complete. Next step: Example 56 — contextual compression to filter what HyDE retrieves.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

### Exercise 1 — Measure the Embedding Distance

**What to look for:** The raw query (`"How does retrieval-augmented generation reduce hallucination?"`) typically retrieves the hallucination document (`"Hallucination occurs when..."`) at rank #1 due to keyword overlap. The hypothesis, which is a longer explanatory paragraph about RAG grounding, often retrieves the RAG grounding document (`"RAG grounds LLM responses..."`) at rank #1 instead — which is the more directly relevant answer.

In [ ]:
# Exercise 1 — sample solution

Q = QUERIES[0]
H = hypotheses[Q]

q_vec = emb_model.embed_query(Q)
h_vec = emb_model.embed_query(H)

q_sims = [(cosine_sim(q_vec, dv), i) for i, dv in enumerate(doc_vecs)]
h_sims = [(cosine_sim(h_vec, dv), i) for i, dv in enumerate(doc_vecs)]

q_ranked = sorted(q_sims, reverse=True)
h_ranked = sorted(h_sims, reverse=True)

print(f"Query: '{Q}'")
print(f"Hypothesis (first 100 chars): {H[:100]}...")
print()
print(f"{'Rank':>4}  {'Q sim':>6}  {'H sim':>6}  Document")
print("-" * 80)

# Sort by query rank; show both scores
h_sim_map = {i: s for s, i in h_sims}
for rank, (q_sim, idx) in enumerate(q_ranked, 1):
    h_sim = h_sim_map[idx]
    better = " ◄ H wins" if h_sim > q_sim else ""
    print(f"  {rank:2d}    {q_sim:.3f}   {h_sim:.3f}  {SAMPLE_DOCS[idx][:55]}{better}")

print()
print(f"Query top-1  : {SAMPLE_DOCS[q_ranked[0][1]]}")
print(f"HyDE  top-1  : {SAMPLE_DOCS[h_ranked[0][1]]}")

### Exercise 2 — Try a Different Hypothesis Prompt

**Expected finding:** The shorter, more directive prompt (`"Answer in 2-3 sentences using technical vocabulary"`) typically produces a more concise hypothesis. Whether it retrieves better documents depends on the query — for this specific third query about HyDE itself, both prompts should find the HyDE document (`"Hypothetical Document Embeddings (HyDE) embeds a generated answer, not the query."`) in the top-1. The key insight is that *any* hypothesis that uses domain vocabulary (`"embedding space"`, `"query-document gap"`, `"bi-encoder"`) will retrieve more relevant documents than the raw short query alone.

In [ ]:
# Exercise 2 — sample solution

ALT_PROMPT = "Answer the following question in 2-3 sentences using technical vocabulary."
Q = QUERIES[2]

def get_top3(prompt_text: str, query: str) -> tuple[str, list[str]]:
    msgs = [SystemMessage(content=prompt_text), HumanMessage(content=query)]
    hyp = llm.invoke(msgs).content
    hyp_vec = emb_model.embed_query(hyp)
    docs = vectorstore.similarity_search_by_vector(hyp_vec, k=3)
    return hyp, [d.page_content for d in docs]

orig_hyp, orig_docs = get_top3(HYPOTHESIS_PROMPT, Q)
alt_hyp,  alt_docs  = get_top3(ALT_PROMPT, Q)

print(f"Query: '{Q}'")
print()
print(f"── Original prompt ({len(orig_hyp)} chars) ──")
print(orig_hyp[:200])
print("Top-3 retrieved:")
for d in orig_docs:
    print(f"  • {d}")

print()
print(f"── Alt prompt ({len(alt_hyp)} chars) ──")
print(alt_hyp[:200])
print("Top-3 retrieved:")
for d in alt_docs:
    print(f"  • {d}")